# Supervised fine-tuning

**Track:** Training LLMs. **Level:** Intermediate. **CPU:** yes.

## Goals

- Format a prompt and a response as one token sequence.
- Mask the loss so it is computed only on the response tokens.
- Overfit a handful of pairs, then try a color the model never trained on.

## Sources and license

Original notebook. The loss mask is the supervised-fine-tuning objective in Ouyang et al., *Training Language Models to Follow Instructions with Human Feedback* (InstructGPT, OpenAI, 2022): predict the answer tokens, not the copied prompt. No InstructGPT data is used. The pairs below were written for this lab.

## Concepts

A causal language model scores every next token. If the prompt is included in the loss, the model is rewarded for reproducing the user's text. SFT sets that reward to zero with a mask. The numerical value of the loss is then the cross-entropy of the answer tokens only. Memorizing eight pairs is a check that the mask and the optimizer are wired up. It is not instruction following. A held-out word that shares a meaning with a training word is the test that separates memorization from the behavior the papers are about.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'torch', 'numpy'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

import math

import torch
import torch.nn as nn

from labs._shared.seed import seed_everything

seed_everything(0)
torch.manual_seed(0)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        if n_embd % n_head != 0:
            raise ValueError("n_embd must be divisible by n_head")
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("mask", mask)

    def forward(self, x):
        b, t, c = x.shape
        qkv = self.qkv(x).view(b, t, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.mask[:, :, :t, :t] == 0, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        mixed = (weights @ v).transpose(1, 2).contiguous().view(b, t, c)
        return self.proj(mixed)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    """A nanoGPT-style decoder. Original code, same block layout as nanoGPT (MIT)."""

    def __init__(self, vocab_size, n_embd=64, n_head=4, n_layer=2, block_size=32):
        super().__init__()
        self.block_size = block_size
        self.tok = nn.Embedding(vocab_size, n_embd)
        self.pos = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList(
            [Block(n_embd, n_head, block_size) for _ in range(n_layer)]
        )
        self.ln = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx, targets=None, loss_mask=None):
        b, t = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(t, device=idx.device))
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln(x))
        loss = None
        if targets is not None:
            per = nn.functional.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
                reduction="none",
            ).view(b, t)
            if loss_mask is None:
                loss = per.mean()
            else:
                loss = (per * loss_mask).sum() / loss_mask.sum().clamp_min(1.0)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n_new, temperature=0.8):
        self.eval()
        for _ in range(n_new):
            logits, _ = self(idx[:, -self.block_size :])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def train_lm(data, vocab_size, steps, n_embd=64, n_layer=2, n_head=4, block_size=32, batch_size=32, lr=3e-3, seed=0):
    g = torch.Generator().manual_seed(seed)
    torch.manual_seed(seed)
    model = TinyGPT(vocab_size, n_embd=n_embd, n_head=n_head, n_layer=n_layer, block_size=block_size)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    for _ in range(steps):
        ix = torch.randint(0, len(data) - block_size - 1, (batch_size,), generator=g)
        x = torch.stack([data[i : i + block_size] for i in ix])
        y = torch.stack([data[i + 1 : i + 1 + block_size] for i in ix])
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        losses.append(float(loss.detach()))
    return model, losses


## Guided exercise: mask and memorization

Each example is `user : <color> assistant : <response>`. The mask is 1 only on the response token. Greedy decoding reads the logits at the last prompt position.


In [ ]:
PAIRS = [
    ("red", "stop"),
    ("green", "go"),
    ("yellow", "wait"),
    ("blue", "sea"),
    ("cat", "meow"),
    ("dog", "bark"),
    ("bird", "sing"),
    ("cow", "moo"),
]
words = ["user", ":", "assistant"] + [w for pair in PAIRS for w in pair] + ["crimson"]
stoi = {word: i for i, word in enumerate(words)}
itos = {i: word for word, i in stoi.items()}

def example(src, dst):
    ids = [stoi["user"], stoi[":"], stoi[src], stoi["assistant"], stoi[":"], stoi[dst]]
    x = torch.tensor(ids[:-1])
    y = torch.tensor(ids[1:])
    mask = torch.tensor([0.0, 0.0, 0.0, 0.0, 1.0])
    return x, y, mask

batch = [example(src, dst) for src, dst in PAIRS]
X = torch.stack([row[0] for row in batch])
Y = torch.stack([row[1] for row in batch])
M = torch.stack([row[2] for row in batch])
assert int(M.sum()) == len(PAIRS)
assert int(M[:, :-1].sum()) == 0

model = TinyGPT(len(stoi), n_embd=32, n_head=2, n_layer=1, block_size=8)
opt = torch.optim.AdamW(model.parameters(), lr=5e-3)
losses = []
for _ in range(250):
    _, loss = model(X, Y, loss_mask=M)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    losses.append(float(loss.detach()))
print("masked loss", round(losses[0], 3), "->", round(losses[-1], 3))
assert losses[-1] < losses[0] * 0.5

@torch.no_grad()
def predict(src: str) -> str:
    ids = torch.tensor([[stoi["user"], stoi[":"], stoi[src], stoi["assistant"], stoi[":"]]])
    logits, _ = model(ids)
    return itos[int(logits[0, -1].argmax())]

correct = sum(predict(src) == dst for src, dst in PAIRS) / len(PAIRS)
print("train exact match", correct, {src: predict(src) for src, _dst in PAIRS})
assert correct >= 0.75
print("LAB OK")


## Your turn

Run the same batch with a mask of all ones, for 30 steps, from a fresh model. The loss should fall. That run is ordinary language-model training on the concatenated string. It is not the SFT objective, because the prompt tokens contribute.

## Solution


In [ ]:
fresh = TinyGPT(len(stoi), n_embd=32, n_head=2, n_layer=1, block_size=8)
opt2 = torch.optim.AdamW(fresh.parameters(), lr=5e-3)
unmasked = []
ones = torch.ones_like(M)
for _ in range(30):
    _, loss = fresh(X, Y, loss_mask=ones)
    opt2.zero_grad(set_to_none=True)
    loss.backward()
    opt2.step()
    unmasked.append(float(loss.detach()))
print("unmasked", round(unmasked[0], 3), "->", round(unmasked[-1], 3))
assert unmasked[-1] < unmasked[0]


## Beyond the course

`crimson` is in the vocabulary and was never a training prompt. The training colors that mean "stop" are only `red`. Report the greedy prediction for `crimson` and the training exact-match rate. A gap is the expected result of this objective on eight memorized pairs.


In [ ]:
heldout = predict("crimson")
print("crimson ->", heldout, "train exact match", correct)
assert heldout in itos.values()


## Research questions

1. The mask throws away the gradient on the prompt. When would you want a small amount of loss on the prompt anyway?
2. Exact match on eight pairs can be 1 while `crimson` is wrong. What data would you add before treating "color to action" as a learned rule?
3. InstructGPT combines this loss with a preference model. Which failure of the mask does the preference model not fix?

## Further reading

- Ouyang et al., *Training Language Models to Follow Instructions with Human Feedback* (OpenAI, 2022).
- Wei et al., *Finetuned Language Models Are Zero-Shot Learners* (FLAN, Google, 2021).
- The SFT loss in Hugging Face TRL (Apache-2.0) is the production version of this mask. Not copied.
